# 📘 智能体架构 2：工具使用

本笔记本介绍第二种智能体架构，也可以说是最具变革性的架构之一：**工具使用**。这一模式是连接大型语言模型推理能力与真实动态世界的桥梁。

没有工具，LLM 是一个封闭系统，受限于冻结在其训练数据中的知识。它无法知道今天的天气、股票的当前价格，或公司数据库中订单的状态。通过赋予智能体使用工具的能力，我们使其能够克服这一根本限制，允许它查询 API、搜索数据库并访问实时信息，以提供不仅经过推理，而且符合事实、及时且相关的答案。

### 定义
**工具使用**架构为基于 LLM 的智能体配备了调用外部函数或 API（即"工具"）的能力。智能体自主决定何时仅靠其内部知识无法回答用户的查询，并确定调用哪个工具来查找必要信息。

### 高层工作流程

1. **接收查询：** 智能体接收来自用户的请求。
2. **决策：** 智能体分析查询及其可用工具。它决定是否需要工具来准确回答问题。
3. **行动：** 如果需要工具，智能体格式化对该工具的调用（例如，具有正确参数的特定函数）。
4. **观察：** 系统执行工具调用，结果（即"观察"）返回给智能体。
5. **综合：** 智能体将工具的输出集成到其推理过程中，为用户生成最终的、有依据的答案。

### 适用场景 / 应用
* **研究助手：** 使用网络搜索 API 回答需要最新信息的问题。
* **企业助手：** 查询内部公司数据库以回答诸如"上周有多少新用户注册？"等问题。
* **科学与数学任务：** 使用计算器或 WolframAlpha 等计算引擎进行 LLM 通常难以处理的精确计算。

### 优缺点
* **优点：**
    * **事实依据：** 通过获取真实、实时的数据大幅减少幻觉。
    * **可扩展性：** 可以简单地添加新工具来持续扩展智能体的能力。
* **缺点：**
    * **集成开销：** 需要仔细的"管道"工作来定义工具、处理 API 密钥和管理潜在的工具故障。
    * **工具信任：** 智能体答案的质量取决于其所使用工具的可靠性和准确性。智能体必须相信其工具提供的信息是正确的。

## 阶段 0：基础与环境设置

与之前一样，我们首先设置环境。这包括安装必要的库以及为 OpenAI、LangSmith 和我们将使用的特定工具配置 API 密钥。

### 步骤 0.1：安装核心库

**我们要做什么：**
我们将安装用于编排（`langchain-openai`、`langgraph`）、环境管理（`python-dotenv`）和打印（`rich`）的标准库集。关键的是，我们还将安装 `tavily-python`，它提供了一个强大的网络搜索工具的简单 API，我们将把这个工具提供给我们的智能体。

In [2]:
# !pip install -q -U langchain-openai langchain langgraph rich python-dotenv tavily-python

### 步骤 0.2：导入库和设置密钥

**我们要做什么：**
我们将导入必要的模块并使用 `python-dotenv` 加载我们的 API 密钥。对于本笔记本，我们需要 OpenAI（用于 LLM）、LangSmith（用于追踪）和 Tavily（用于网络搜索工具）的密钥。

**需要的操作：** 在此目录中创建一个包含密钥的 `.env` 文件：
```
OPENAI_API_KEY="your_openai_api_key_here"
OPENAI_API_BASE_URL="your_openai_api_base_url_here"
LANGCHAIN_API_KEY="your_langsmith_api_key_here"
TAVILY_API_KEY="your_tavily_api_key_here"
```

In [3]:
import os
import json
from typing import List, Annotated, TypedDict, Optional
from dotenv import load_dotenv

# LangChain components
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import BaseMessage, ToolMessage
from pydantic import BaseModel, Field

# LangGraph components
from langgraph.graph import StateGraph, END
from langgraph.graph.message import AnyMessage, add_messages
from langgraph.prebuilt import ToolNode

# For pretty printing
from rich.console import Console
from rich.markdown import Markdown

# --- API Key and Tracing Setup ---
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Agentic Architecture - Tool Use (OpenAI)"

# Check that the keys are set
for key in ["OPENAI_API_KEY", "LANGCHAIN_API_KEY", "TAVILY_API_KEY"]:
    if not os.environ.get(key):
        print(f"{key} not found. Please create a .env file and set it.")

print("Environment variables loaded and tracing is set up.")

Environment variables loaded and tracing is set up.


## 阶段 1：定义智能体的工具包

智能体的能力取决于其可以访问的工具。在这个阶段，我们将定义和测试我们将提供给智能体的特定工具：实时网络搜索。

### 步骤 1.1：创建和测试网络搜索工具

**我们要做什么：**
我们将实例化 `TavilySearchResults` 工具。定义工具最关键的部分是其**描述**。LLM 使用这个自然语言描述来理解工具的作用以及何时应该使用它。清晰、精确的描述对于智能体做出正确决策至关重要。然后我们将直接测试工具，看看其原始输出是什么样的。

In [4]:
# Initialize the tool. We can set the max number of results to keep the context concise.
search_tool = TavilySearchResults(max_results=2)

# It's crucial to give the tool a clear name and description for the agent
search_tool.name = "web_search"
search_tool.description = "A tool that can be used to search the internet for up-to-date information on any topic, including news, events, and current affairs."

tools = [search_tool]
print(f"Tool '{search_tool.name}' created with description: '{search_tool.description}'")

console = Console()

# Let's test the tool directly to see its output format
print("\n--- Testing the tool directly ---")
test_query = "What was the score of the last Super Bowl?"
test_result = search_tool.invoke({"query": test_query})
console.print(f"[bold green]Query:[/bold green] {test_query}")
console.print("\n[bold green]Result:[/bold green]")
console.print(test_result)

Tool 'web_search' created with description: 'A tool that can be used to search the internet for up-to-date information on any topic, including news, events, and current affairs.'

--- Testing the tool directly ---


C:\Users\guodp\AppData\Local\Temp\ipykernel_1268\1362582966.py:2: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(max_results=2)


Query: What was the score of the last Super Bowl?

Result:

[
    {
        'title': 'Super Bowl Scores Last 10 Years | StatMuse',
        'url': 'https://www.statmuse.com/nfl/ask/super-bowl-scores-last-10-years',
        'content': 'Sign in/up\n\n NFL\n CFB\n NBA\n FC\n NHL\n MLB\n WNBA\n PGA\n Money\n\n Trending Sports\n 
Trending Money\n Trending Live\n\n Data & Glossary\n\nSign in/up\n\nSign in/up\n\n Home\n\n NFL\n CFB\n NBA\n FC\n 
NHL\n MLB\n WNBA\n PGA\n Money\n\n Scores\n\n Trending\n\n Trending Sports\n Trending Money\n Trending Live\n\n 
Examples\n\n Data & Glossary\n\n Gallery\n\n About\n\n Blog\n\n Shop\n\n# The Philadelphia Eagles dominated the 
Kansas City Chiefs, 40 to 22, in Super Bowl LIX on February 9.\n\nSun, Feb 9, 2025\n\nSun, Feb 11, 
2024\n\n22\n\nChiefs\n\nFinal\n\nSun, Feb 12, 2023\n\nChiefs\n\nEagles\n\nFinal\n\nSun, Feb 13, 
2022\n\nBengals\n\n20\n\nFinal\n\nSun, Feb 7, 2021\n\nChiefs\n\n9\n\nBuccaneers\n\n31\n\nFinal\n\nSun, Feb 2, 
2020\n\n49ers\n\n20\n\nChiefs\n\n31\n\nFinal\n\nSun, Feb 3, 2019\n\nPatriots\n\n13\n\nRams\n\n3\n\nFinal\n\nSun, 
Feb 4, 2018\n\nEagles\n\n41\n\nPatriots\n\n33\n\nFinal\n\nSun, Feb 5, 2017 [...] 
5-12-1\n\n6-11\n\n6-11\n\n6-11\n\n7-10\n\n## NFL 2025 Standings\n\n| AFC East | W | L | T | PCT |\n ---  --- \n| 
Patriots - y | 14 | 3 | 0 | .824 |\n| Bills - w | 12 | 5 | 0 | .706 |\n| Dolphins - e | 7 | 10 | 0 | .412 |\n| Jets
- e | 3 | 14 | 0 | .176 |\n| AFC North | W | L | T | PCT |\n| Steelers - y | 10 | 7 | 0 | .588 |\n| Ravens - e | 8 
| 9 | 0 | .471 |\n| Bengals - e | 6 | 11 | 0 |  |\n| Browns - e | 5 | 12 | 0 | .294 |\n| AFC South | W | L | T | 
PCT |\n| Jaguars - y | 13 | 4 | 0 |  |\n| Texans - w | 12 | 5 | 0 | .706 |\n| Colts - e | 8 | 9 | 0 | .471 |\n| 
Titans - e | 3 | 14 | 0 | .176 |\n| AFC West | W | L | T | PCT |\n| Broncos - z\\ | 14 | 3 | 0 | .824 |\n| Chargers
- w | 11 | 6 | 0 | .647 |\n| Chiefs - e | 6 | 11 | 0 | .353 |\n| Raiders - e | 3 | 14 | 0 | .176 | [...] 
Eagles\n\n41\n\nPatriots\n\n33\n\nFinal\n\nSun, Feb 5, 2017\n\nPatriots\n\n34\n\nFalcons\n\n28\n\nFinal\n\nSun, Feb
7, 2016\n\nPanthers\n\n10\n\nBroncos\n\n24\n\nFinal\n\n### Related Searches\n\n What team has the worst record as a
favorite since 2018?\n Who is the all-time Super Bowl passing leader?\n Who threw the most touchdowns in the 
1990s?\n See trending\n\n## More Eagles Stats\n\nTeam Leaders  \n\nPASS\n\n### 168\n\nHurts\n\nRUSH\n\n### 
106\n\nBarkley\n\nREC\n\n### 70\n\nSmith\n\nTeam Rankings \n\nPPG\n\n22.3\n\n19th\n\nOPP PPG\n\n19.1\n\n5th\n\nRUSH
YDS/G\n\n116.9\n\n18th\n\nPASS YDS/G\n\n205.8\n\n23rd\n\n2025 Division Standings \n\n| TEAM | W | L | T | PCT |\n 
---  --- \n| Eagles | 11 | 6 | 0 | .647 |\n| Cowboys | 7 | 9 | 1 | .441 |\n| Commanders | 5 | 12 | 0 | .294 |\n| 
Giants | 4 | 13 | 0 | .235 |\n\nLast Game \n\n49ers\n\n1 - 0\n\n23\n\nFinal\n\n19\n\nEagles\n\n0 - 1\n\nSun 11 Jan 
2026',
        'score': 0.7089592
    },
    {
        'title': 'Eagles 40-22 Chiefs (Feb 9, 2025) Final Score - ESPN',
        'url': 'https://www.espn.com/nfl/game/_/gameId/401671889/chiefs-eagles',
        'content': "## 2025 Standings\n\nAmerican Football Conference\n\n| AFC West | W | L | T | PCT | PF | PA |\n
---  ---  --- \n| Denver | 14 | 3 | 0 | .824 | 401 | 311 |\n| Los Angeles | 11 | 6 | 0 | .647 | 368 | 340 |\n| 
Kansas City | 6 | 11 | 0 | .353 | 362 | 328 |\n| Las Vegas | 3 | 14 | 0 | .176 | 241 | 432 |\n\nNational Football 
Conference\n\n| NFC East | W | L | T | PCT | PF | PA |\n ---  ---  --- \n| Philadelphia | 11 | 6 | 0 | .647 | 379 |
325 |\n| Dallas | 7 | 9 | 1 | .441 | 471 | 511 |\n| Washington | 5 | 12 | 0 | .294 | 356 | 451 |\n| New York | 4 | 
13 | 0 | .235 | 381 | 439 |\n\n## NFL News\n\n## Patriots QB Maye misses practice with illness, Vrabel 
says\n\nPatriots coach Mike Vrabel says quarterback Drake Maye didn't practice Friday because of illness.## 
Seahawks-Patriots Super Bowl history: Records, stats, facts",
        'score': 0.528169
    }
]

**输出讨论：**
测试显示了我们的 `web_search` 工具的原始输出。它返回一个字典列表，其中每个字典包含搜索结果的 URL 和内容片段。这种结构化信息正是智能体在决定使用工具后将作为其"观察"接收的内容。现在我们有了一个可用的工具，可以构建将学习如何使用它的智能体。

## 阶段 2：使用 LangGraph 构建工具使用智能体

现在我们将构建智能体工作流程。这涉及使 LLM 意识到工具，并创建一个允许它循环通过"思考-行动-观察"周期的图，这是工具使用的本质。

### 步骤 2.1：定义图状态

**我们要做什么：**
工具使用智能体的状态通常是一个代表对话历史记录的消息列表。此历史记录包括用户的问题、智能体的思考和工具调用，以及来自这些工具的结果。我们将使用一个可以容纳任何类型 LangChain 消息的 `TypedDict`。

In [5]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

print("AgentState TypedDict defined to manage conversation history.")

AgentState TypedDict defined to manage conversation history.


### 步骤 2.2：将工具绑定到 LLM

**我们要做什么：**
这是使 LLM"感知"工具的关键步骤。我们使用 `.bind_tools()` 方法，它将我们工具的名称和描述传递给 LLM 的系统提示。这使得模型的内部逻辑能够根据描述决定何时调用工具。

In [6]:
model = os.environ.get("OPENAI_API_MODEL", "gpt-4o")
base_url = os.environ.get("OPENAI_API_BASE_URL", "https://api.openai.com/v1")
llm = ChatOpenAI(model=model, base_url=base_url, temperature=0)

# Bind the tools to the LLM, making it tool-aware
llm_with_tools = llm.bind_tools(tools)

print("LLM has been bound with the provided tools.")

LLM has been bound with the provided tools.


### 步骤 2.3：定义智能体节点

**我们要做什么：**
我们的图将有两个主要节点：
1. **`agent_node`：** 这是"大脑"。它使用当前对话历史记录调用 LLM。LLM 的响应要么是最终答案，要么是调用工具的请求。
2. **`tool_node`：** 这是"手"。它接收来自 `agent_node` 的工具调用请求，执行相应的工具，并返回输出。我们将使用 LangGraph 预构建的 `ToolNode` 来实现此功能。

In [7]:
def agent_node(state: AgentState):
    """The primary node that calls the LLM to decide the next action."""
    console.print("--- AGENT: Thinking... ---")
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# The ToolNode is a pre-built node from LangGraph that executes tools
tool_node = ToolNode(tools)

print("Agent node and Tool node have been defined.")

Agent node and Tool node have been defined.


### 步骤 2.4：定义条件路由器

**我们要做什么：**
在 `agent_node` 运行后，我们需要决定接下来去哪里。路由器函数检查来自智能体的最后一条消息。如果该消息包含 `tool_calls` 属性，这意味着智能体想要使用工具，所以我们路由到 `tool_node`。如果没有，这意味着智能体有最终答案，我们可以结束工作流程。

In [8]:
def router_function(state: AgentState) -> str:
    """Inspects the agent's last message to decide the next step."""
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        # The agent has requested a tool call
        console.print("--- ROUTER: Decision is to call a tool. ---")
        return "call_tool"
    else:
        # The agent has provided a final answer
        console.print("--- ROUTER: Decision is to finish. ---")
        return "__end__"

print("Router function defined.")

Router function defined.


## 阶段 3：组装和运行工作流程

现在我们将把所有组件连接成一个完整的、可执行的图，并在一个迫使智能体使用其新网络搜索能力的查询上运行它。

### 步骤 3.1：构建和可视化图

**我们要做什么：**
我们将创建 `StateGraph` 并添加我们的节点和边。关键部分是使用我们的 `router_function` 创建智能体的主要推理循环的条件边：`agent -> router -> tool -> agent`。

In [9]:
graph_builder = StateGraph(AgentState)

# Add the nodes
graph_builder.add_node("agent", agent_node)
graph_builder.add_node("call_tool", tool_node)

# Set the entry point
graph_builder.set_entry_point("agent")

# Add the conditional router
graph_builder.add_conditional_edges(
    "agent",
    router_function,
)

# Add the edge from the tool node back to the agent to complete the loop
graph_builder.add_edge("call_tool", "agent")

# Compile the graph
tool_agent_app = graph_builder.compile()

print("Tool-using agent graph compiled successfully!")

# Visualize the graph
try:
    from IPython.display import Image, display
    png_image = tool_agent_app.get_graph().draw_png()
    display(Image(png_image))
except Exception as e:
    print(f"Graph visualization failed: {e}. Please ensure pygraphviz is installed.")

Tool-using agent graph compiled successfully!
Graph visualization failed: Install pygraphviz to draw graphs: `pip install pygraphviz`.. Please ensure pygraphviz is installed.


**输出讨论：**
编译好的图已准备就绪。可视化清楚地显示了智能体的推理循环。过程从 `agent` 节点开始。然后条件边（由菱形表示）路由流程。如果需要工具，它转到 `call_tool`，输出反馈给 `agent` 进行综合。如果不需要工具，过程转到 `__end__`。这个结构完美地实现了工具使用模式。

### 步骤 3.2：端到端执行

**我们要做什么：**
让我们运行智能体，提出一个它无法从训练数据中知道的问题，迫使它使用网络搜索工具。我们将流式传输中间步骤以观察其推理过程的展开。

In [10]:
user_query = "What were the main announcements from Apple's latest WWDC event?"
initial_input = {"messages": [("user", user_query)]}

console.print(f"[bold cyan]🚀 Kicking off Tool Use workflow for request:[/bold cyan] '{user_query}'\n")

for chunk in tool_agent_app.stream(initial_input, stream_mode="values"):
    chunk["messages"][-1].pretty_print()
    console.print("\n---\n")

console.print("\n[bold green]✅ Tool Use workflow complete![/bold green]")

🚀 Kicking off Tool Use workflow for request: 'What were the main announcements from Apple's latest WWDC event?'

================================ Human Message =================================

What were the main announcements from Apple's latest WWDC event?


---

--- AGENT: Thinking... ---

--- ROUTER: Decision is to call a tool. ---

...

--- AGENT: Thinking... ---

--- ROUTER: Decision is to finish. ---

================================== Ai Message ==================================

Apple’s latest WWDC keynote (WWDC25, June 2025) was centered on a cross‑platform design overhaul, expanded Apple Intelligence features, and major OS updates—plus a new gaming hub.

## Biggest cross‑platform announcements
- **Unified OS versioning:** Apple aligned naming across platforms to **“26”** (iOS 26, iPadOS 26, watchOS 26, tvOS 26, visionOS 26, and **macOS Tahoe 26**).
- **New design language: “Liquid Glass”** across all Apple operating systems (new translucent/material look spanning system UI, app chrome, controls, etc.).
- **Apple Intelligence expands** with more built-in capabilities and deeper system integration, including:
  - **Live Translation** built into **Messages, FaceTime, and Phone** (real-time text/audio translation, with on-device processing emphasized for privacy).
  - A developer-facing **Foundation Models framework** to access Apple’s **on-device** model for private, offline-capab

---

✅ Tool Use workflow complete!

## 阶段 4：评估

现在智能体已经运行，我们可以评估其性能。对于工具使用智能体，我们关心两件事：它是否正确使用了工具，以及从工具输出综合得出的最终答案是否高质量？

### 步骤 4.1：分析执行追踪

**我们要做什么：**
通过查看上一步的流式输出，我们可以追踪智能体的确切思考过程。输出显示了流经图状态的不同消息类型（带有 `tool_calls` 的 `AIMessage`、带有结果的 `ToolMessage`）。

**输出讨论：**
执行追踪清楚地显示了工具使用的实际运作：
1. 打印的第一条消息来自 `agent` 节点。它是一条包含 `tool_calls` 属性的 `AIMessage`，表明 LLM 正确决定使用 `web_search` 工具。
2. 下一条消息是 `ToolMessage`。这是 `tool_node` 在执行搜索并返回原始结果后的输出。
3. 最后一条消息是另一条 `AIMessage`，但这次没有 `tool_calls`。这是智能体将来自 `ToolMessage` 的信息综合成对用户的连贯最终答案。

此追踪确认了智能体的逻辑和图的路由完美工作。

### 步骤 4.2：使用 LLM 作为评判者进行评估

**我们要做什么：**
我们将创建一个"评判者" LLM 来提供对智能体性能的结构化、定量评估。评估标准将专门针对评估工具使用的质量。

In [ ]:
class ToolUseEvaluation(BaseModel):
    """Schema for evaluating the agent's tool use and final answer."""
    tool_selection_score: int = Field(description="Score 1-5 on whether the agent chose the correct tool for the task.")
    tool_input_score: int = Field(description="Score 1-5 on how well-formed and relevant the input to the tool was.")
    synthesis_quality_score: int = Field(description="Score 1-5 on how well the agent integrated the tool's output into its final answer.")
    justification: str = Field(description="A brief justification for the scores.")

judge_llm = llm.with_structured_output(ToolUseEvaluation)

# To evaluate, we need to reconstruct the full conversation trace
final_answer = tool_agent_app.invoke(initial_input)
conversation_trace = "\n".join([f"{m.type}: {m.content or ''} {getattr(m, 'tool_calls', '')}" for m in final_answer['messages']])

def evaluate_tool_use(trace: str):
    prompt = f"""You are an expert judge of AI agents. Evaluate the following conversation trace based on the agent's tool use on a scale of 1-5. Provide a brief justification.
    
    Conversation Trace:
    ```
    {trace}
    ```
    """
    return judge_llm.invoke(prompt)

console.print("--- Evaluating Tool Use Performance ---")
evaluation = evaluate_tool_use(conversation_trace)
console.print(evaluation.model_dump())

--- AGENT: Thinking... ---

--- ROUTER: Decision is to call a tool. ---

--- AGENT: Thinking... ---

--- ROUTER: Decision is to call a tool. ---

--- AGENT: Thinking... ---

...

--- AGENT: Thinking... ---

--- ROUTER: Decision is to finish. ---

--- Evaluating Tool Use Performance ---

{
    'tool_selection_score': 4,
    'tool_input_score': 4,
    'synthesis_quality_score': 4,
    'justification': 'Tool choice (web_search) was appropriate for a timely, factual summary of WWDC announcements,
and the agent cross-checked across reputable sources (Apple Newsroom, The Verge, CNET, Apple ML). However, tool use
was somewhat excessive/fragmented (many separate searches where 1–2 comprehensive roundups plus Apple press 
releases could suffice), and a few sources were lower-signal (YouTube commentary). Queries were generally 
well-targeted and specific. The final response synthesized the retrieved information into a clear, structured list 
of “main announcements,” though it mixed keynote items with broader WWDC/developer-details and included a couple of
potentially debatable inclusions as ‘main’ items (e.g., Xcode model specifics), without clearly distinguishing 
keynote highlights vs. ancillary details.'
}

: 

**输出讨论：**
LLM 作为评判者提供了对我们智能体性能的结构化和有根据的评估。在所有三个类别中的高分——`tool_selection_score`、`tool_input_score` 和 `synthesis_quality_score`——证实了我们的智能体不仅在使用工具，而且*有效地*使用它们。它正确识别了网络搜索的需求，制定了相关查询，并成功将检索到的事实综合成有帮助且准确的最终答案。这种自动评估使我们对实现的稳健性充满信心。

## 结论

在本笔记本中，我们基于**工具使用**架构构建了一个完整的、可运行的智能体。我们成功地为 OpenAI 驱动的 LLM 配备了网络搜索工具，并使用 LangGraph 创建了一个健壮的推理循环，允许智能体决定何时以及如何使用它。

端到端执行和随后的评估证明了这一模式的巨大价值。通过将我们的智能体连接到实时的外部信息，我们从根本上克服了静态训练数据的限制。智能体不再仅仅是一个推理者；它是一个研究者，能够提供有依据、符合事实且及时的答案。这一架构是创建几乎任何实用的、现实世界 AI 助手的基础构建块。